## Dataset up 

In [ ]:
include("main_utils.jl")
include("data_setup.jl")
include("comix_uk_time_series.jl")

default_plot_setting()

In [ ]:
df, df_part = read_raw_sc_data_with_sday("comix_uk");
df, df_part = read_adult_chunks(df, df_part);

In [ ]:
df_dds = create_df_dds_chunk(df, df_part)
CSV.write("../dt_intermediate/df_dds.csv", df_dds)

## BNB for the later period

In [ ]:
include("main_utils.jl")
include("data_setup.jl")
include("comix_uk_time_series.jl")

In [ ]:
df_dds = @pipe CSV.read("../dt_intermediate/df_dds.csv", DataFrame) |>
    @rename(_, :date = :key) |>
    @transform(_, :key = string.(:date)) |>
    @subset(_, :strat .== "non-home");
df_dds_late = @subset(df_dds, :date .>= Date(2021, 7, 1));
dds_nhm_late = df_dds_late |> merge_dd

In [ ]:
#chn = fit_model_with_forward_mode(model_BNB2(dds_nhm_late), 1000)
#jldsave("../dt_intermediate/chn_bnb2_nhm_late.jld2", result=chn);

In [ ]:
#chn = fit_model_with_forward_mode(model_ZeroInfBNB2(dds_nhm_late), 1000)
#jldsave("../dt_intermediate/chn_zeroinf_bnb2_nhm_late.jld2", result=chn);

In [ ]:
chn = load("../dt_intermediate/chn_zeroinf_bnb2_nhm_late.jld2", "result")
plot(chn)

In [ ]:
calc_waic(get_vec_ZeroInfBNB2_from_chn(chn), dds_nhm_late)
# BNB: 187813.2473621587
# BNB2: 187345.69795166497

In [ ]:
bnb = get_ZeroInfBNB2(chn)

In [ ]:
pl = plot(xlim=[0,30], ylim=[-5, 0])
plot_pdf!(pl, dds_nhm_late;
    markersize=0.5, markerstrokewidth=0.1, label=lbl)
plot_pdf!(pl, bnb)

In [ ]:
include("main_utils.jl")

In [ ]:
lbl = "Non-home, July-2021 ~ Mar-2022"
pl = plot(xaxis=:log10)
plot_ccdf!(pl, dds_nhm_late;
    markersize=0.5, markerstrokewidth=0.1, label=lbl)
#plot_ccdf!(pl, zero_trunc(bnb)
plot_ccdf!(pl, bnb)

## Bi-weekly fitting

In [ ]:
# Bi-weekly ZeroInfBNB2 fits — non-home contacts
# Skip-if-exists so re-running is cheap
dir_ = "../dt_intermediate"
keys_ = sort(unique(df_dds.key))

for k in keys_
    path = "$(dir_)/$(k)_zeroinf_bnb2_nhm.jld2"
    if isfile(path)
        println("Already fitted, skipping: $k")
        continue
    end
    dd = @subset(df_dds, :key .== k, :strat .== "non-home") |> DegreeDist
    println("Fitting: $k  (n = $(sum(dd.y)))")
    chn = fit_model_with_forward_mode(model_ZeroInfBNB2(dd), 1000; progress=false)
    jldsave(path, result=chn)
    println("  ✓ saved → $path")
end
println("All waves done.")

In [ ]:
# Extract posterior median mean + α, with 95% CI from each wave chain
dir_ = "../dt_intermediate"
rows = []
for k in keys_
    path = "$(dir_)/$(k)_zeroinf_bnb2_nhm.jld2"
    chn = load(path, "result")
    d_med = get_ZeroInfBNB2(chn)
    m_med = mean(d_med)
    α_med = d_med.v + 1   # BNB2: v = α - 1

    dists = get_vec_ZeroInfBNB2_from_chn(chn)
    ms = mean.(dists)
    αs = map(d -> d.v + 1, dists)

    m_l, m_u = quantile(ms, [0.025, 0.975])
    α_l, α_u = quantile(αs, [0.025, 0.975])

    push!(rows, (key=Date(k), mean=m_med, mean_l=m_l, mean_u=m_u,
                              α=α_med, α_l=α_l, α_u=α_u))
end

df_means = DataFrame(rows)
sort!(df_means, :key)
display(df_means)

In [ ]:
# Temporal mean and tail-heaviness (α) with 95% posterior CI
pl_mean = plot(
    df_means.key, df_means.mean;
    ribbon = (df_means.mean .- df_means.mean_l, df_means.mean_u .- df_means.mean),
    fillalpha = 0.3,
    ylabel = "Mean contacts",
    label = "ZeroInfBNB2 posterior median",
    title = "Bi-weekly non-home contacts",
    xrotation = 45,
    marker = :circle, markersize = 4,
    legend = :topright,
)

pl_α = plot(
    df_means.key, df_means.α;
    ribbon = (df_means.α .- df_means.α_l, df_means.α_u .- df_means.α),
    fillalpha = 0.3,
    xlabel = "Date",
    ylabel = "α (tail index)",
    label = "α = v + 1",
    xrotation = 45,
    marker = :circle, markersize = 4,
    legend = :topright,
    color = :crimson,
    ylim = [0.9, 2.5]
)
# α < 2 → infinite variance; α < 1 → infinite mean — mark these thresholds
hline!(pl_α, [1.0]; linestyle = :dash, color = :grey, label = "α = 1")
hline!(pl_α, [2.0]; linestyle = :dot,  color = :grey, label = "α = 2")

plot(pl_mean, pl_α; layout = (2, 1), size = (800, 600), left_margin = 5Plots.mm)

In [ ]:
# CCDF fit diagnostics — 4 bi-weekly waves overlaid per panel
dir_ = "../dt_intermediate"
group_size = 4
key_groups = [keys_[i:min(i+group_size-1, end)] for i in 1:group_size:length(keys_)]
palette4 = [1,2,3,4] #palette(:tab10, [1,2,3,4])

subplots = []
for grp in key_groups
    pl = plot(; xaxis=:log10, ylim=[-4, 0], xlim=[1, 10_000],
                legend=:bottomleft, #legendfontsize=18,
                xlabel="Contacts", ylabel="log10 CCDF")
    for (ci, k) in enumerate(grp)
        dd = @subset(df_dds, :key .== k) |> DegreeDist
        chn = load("$(dir_)/$(k)_zeroinf_bnb2_nhm.jld2", "result")
        d_fit = get_ZeroInfBNB2(chn)
        col = palette4[ci]
        plot_ccdf!(pl, dd;
            markersize=1.5, markerstrokewidth=0.0,
            color=col, label=k)
        plot_ccdf!(pl, d_fit;
            color=col, linewidth=1.5, linestyle=:dash, label="")
    end
    push!(subplots, pl)
end

ncols = 2
nrows = ceil(Int, length(key_groups) / ncols)
plot(subplots...;
    layout=(nrows, ncols),
    size=(ncols * 420, nrows * 320),
    left_margin=4Plots.mm, bottom_margin=4Plots.mm,
)

In [ ]:
chn = load("../dt_intermediate/2022-02-25_zeroinf_bnb2_nhm.jld2", "result")
describe(chn)

In [ ]:
plot(chn)